# CNN-BN 다중 seed 강건성 검증

notebook 09의 CNN-BN 단일 seed 결과가 우연이 아닌지 확인한다.  id_00, -6dB 데이터에서 seed 5개로 반복 학습하고 AUC 평균과 표준편차를 계산한다.

- 비교 기준: Linear AutoEncoder `0.9656 ± 0.0114`
- 통제 조건: 기존 하이퍼파라미터와 `src` 함수 그대로 사용
- 상세 설계: `10_cnnbn_robustness_설계.md`


In [1]:
## [0] 준비
# 노트북은 notebooks 폴더에서 실행되므로, 프로젝트 루트를 import 경로에 추가한다.
import sys
sys.path.insert(0, '..')

# torch와 다른 라이브러리의 OpenMP 런타임 충돌을 막는다.
# 반드시 torch를 import하기 전에 환경 변수를 설정해야 한다.
import os
os.environ['KMP_DUPLICATE_LIB_OK'] = 'TRUE'

import numpy as np
import torch
from src import data, train, evaluate
from src.model import CNNBottleneckAE

# CUDA를 사용할 수 있으면 GPU로 학습하고, 아니면 CPU를 사용한다.
DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
print('학습 장치 :', DEVICE)


학습 장치 : cuda


In [2]:
## [1] CNN-BN을 여러 seed로 반복 학습하고 AUC 계산
# notebook 07의 Linear 실험과 같은 seed를 사용해야 공정하게 비교할 수 있다.
SEEDS = [0, 1, 2, 3, 4]
aucs = []

for seed in SEEDS:
    # 모델 초기값과 DataLoader 셔플 순서를 현재 seed로 고정한다.
    torch.manual_seed(seed)

    # 기본 경로는 id_00, -6dB 데이터다.
    # 같은 seed로 정상 데이터를 학습 80%, 테스트 20%로 분할한다.
    trainNP, teN, teA = data.load_split_normalize(seed=seed)

    # seed마다 새로운 CNN-BN 모델을 만들고 정상 데이터만 학습한다.
    model = CNNBottleneckAE().to(DEVICE)
    model, _ = train.train_model(model, trainNP, DEVICE)

    # 테스트 정상과 테스트 이상의 샘플별 복원 오차를 각각 계산한다.
    err_normalNP = evaluate.recon_errors(model, teN, DEVICE)
    err_abnormalNP = evaluate.recon_errors(model, teA, DEVICE)

    # 이상 샘플의 복원 오차가 정상보다 큰 정도를 ROC AUC로 계산한다.
    auc = evaluate.compute_auc(err_normalNP, err_abnormalNP)
    aucs.append(auc)
    print(f'seed {seed}: AUC {auc:.4f}')


seed 0: AUC 0.9774
seed 1: AUC 0.9614
seed 2: AUC 0.9819
seed 3: AUC 0.9664
seed 4: AUC 0.9772


In [3]:
## [2] 5개 seed 결과의 평균과 표준편차
# 계산을 위해 Python 리스트를 NumPy 배열로 변환한다.
aucsNP = np.array(aucs)

# notebook 07과 동일하게 np.std()로 모집단 표준편차를 계산한다.
AUC_MEAN = aucsNP.mean()
AUC_STD = aucsNP.std()

# Linear 결과(0.9656 ± 0.0114)와 바로 비교할 수 있는 형식으로 출력한다.
print(f'AUC 평균 {AUC_MEAN:.4f} ± {AUC_STD:.4f}  (seed {len(aucsNP)}개)')


AUC 평균 0.9728 ± 0.0077  (seed 5개)


In [ ]:
## [3] CNN-BN seed별 성능과 Linear 기준 비교
# CNN-BN의 실행별 AUC와 Linear 다중 seed 평균·범위를 한 그래프에서 비교한다.
import matplotlib.pyplot as plt
import koreanize_matplotlib

LINEAR_MEAN = 0.9656
LINEAR_STD = 0.0114

plt.figure(figsize=(8, 4))
plt.plot(SEEDS, aucsNP, marker='o', linewidth=2, label='CNN-BN AUC')
plt.axhline(AUC_MEAN, color='tab:blue', linestyle='--', label=f'CNN-BN 평균 {AUC_MEAN:.4f}')
plt.axhline(LINEAR_MEAN, color='tab:orange', linestyle='--', label=f'Linear 평균 {LINEAR_MEAN:.4f}')
plt.axhspan(LINEAR_MEAN-LINEAR_STD, LINEAR_MEAN+LINEAR_STD, color='tab:orange', alpha=0.15, label='Linear 평균 ± 표준편차')
plt.xticks(SEEDS)
plt.ylim(0.93, 1.00)
plt.xlabel('시드')
plt.ylabel('AUC')
plt.title('CNN-BN 시드별 안정성과 Linear 기준 비교')
plt.grid(axis='y', alpha=0.3)
plt.legend()
plt.show()


## 결과 해석

| seed | CNN-BN AUC |
|:----:|:----------:|
| 0 | 0.9774 |
| 1 | 0.9614 |
| 2 | 0.9819 |
| 3 | 0.9664 |
| 4 | 0.9772 |

- CNN-BN: **0.9728 ± 0.0077**
- Linear: **0.9656 ± 0.0114**
- CUDA 비결정성 때문에 임시 실행값 `0.9720 ± 0.0073`과 소수점 수준 차이가 있지만 결론은 같다.
- 두 모델의 표준편차 범위가 겹치므로 CNN-BN을 Linear 대비 **동급~소폭 우위**로 해석한다.
